# Where does all the GPU memory go when training an LLM?

Whenever I read articles about LLM training, I keep coming back to the same question. Much of the challenge seems to be around memory capacity and bandwidth rather than compute itself.

I'd actually be happy to be compute-bound. At least that would tell me I'm making good use of the compute I'm paying for.

So the question is: **Where does all the GPU memory go when training an LLM?**

As a senior software engineer, I prefer to see how things work under the hood. It helps me make better engineering decisions when new requirements come along.

Think about how we approach an unfamiliar legacy codebase. We read the code, trace the execution flow, set a few breakpoints, and debug the parts we don't fully understand. Eventually, we build a mental model of how the system works.

But how do we do something similar with a Transformer?

My approach here is a little different from the existing articles and documentation on training memory, many of which already explain the theory quite well.

I want to take a hands-on approach. We'll use TransformerLens to look inside a Transformer, see how individual activations are created during the forward pass, inspect their shapes, and understand how much GPU memory they consume.

Rather than starting with a memory formula and accepting the numbers, let's open up the model and see where those numbers come from.

**A quick note on TransformerLens:** Originally developed by Neel Nanda, TransformerLens is a library designed for mechanistic interpretability — understanding how neural networks perform computations internally. We'll use it for a slightly different purpose here: investigating memory consumption.

If you're interested in mechanistic interpretability, check out the [TransformerLens GitHub repository](https://github.com/TransformerLensOrg/TransformerLens). That's a fascinating subject on its own, and perhaps something I'll cover in a future post.

We are using nvidia-smi to get details about the GPU we are using and how much memory it has. We are intentionally using the T4 because Google Colab offers it for free, subject to availability, so anyone can run this exercise without paying for GPU compute.

In [ ]:
# Let view what there in the device(GPU) memory
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu

In [ ]:
import torch
# Nothing to do with Tranformer , just triggers the CUDO context + kernel loads
# so we can mesure with less noise.
x = torch.zeros(1, device="cuda")
x = x @ x
torch.cuda.synchronize()

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu

### Initialize the GPT-2 Model and Load the Pretrained Weights

Let's initialize GPT-2 with pretrained weights and observe how much GPU memory it occupies.

### Model Weights

The first thing we load into GPU memory is the model's weights — the learned parameters.

In one GPT-2 transformer block, the main weights are attention (`Wq`, `Wk`, `Wv`, `Wo`) and feed-forward (`Wupff`, `Wdownff`), along with biases and LayerNorm parameters.

With hidden dimension `d_model`:

- Attention: 4 × d_model²
- Feed-forward: 8 × d_model²
- Total per block: 12 × d_model² parameters

This doesn't count biases and normalization parameters, and it assumes the standard GPT-2 feed-forward expansion ratio of 4. The full model also has token embeddings, positional embeddings, and a final output layer.

**For GPT-2 small (d_model = 768, L = 12 layers), this works out to roughly 124M parameters in total. That number will come back later when we build a full memory budget.**

If we load the model in:

- **FP32:** 124M × 4 bytes = 496 MB (~0.5 GB)
- **FP16/BF16:** 124M × 2 bytes = 248 MB (~0.25 GB)

This is what we expect based on our calculations. Let's load the model and see how much memory it actually occupies.

In [ ]:
from collections import defaultdict
from transformers import AutoModelForCausalLM

# you can change the dtype to f16 or bf16 or int8
model = AutoModelForCausalLM.from_pretrained("gpt2", dtype=torch.float32).to("cuda").eval()

# named_parameters() de-duplicates tied weights by default,
# so lm_head is NOT double-counted.
groups = defaultdict(int)
for name, p in model.named_parameters():
    if name.endswith("wte.weight"):      k = "token embedding (wte)"
    elif name.endswith("wpe.weight"):    k = "position embedding (wpe)"
    elif "lm_head" in name:              k = "unembedding (lm_head, TIED)"
    elif ".attn." in name:               k = "attention (per layer sum)"
    elif ".mlp."  in name:               k = "MLP (per layer sum)"
    elif "ln" in name:                   k = "LayerNorm"
    else:                                k = "other"
    groups[k] += p.numel()

total = sum(groups.values())
print(f"{'component':32s} {'params':>10s} {'FP32 MiB':>10s}")
for k, v in groups.items():
    print(f"{k:32s} {v/1e6:10.2f}M {v*4/1024**2:10.1f}")
print(f"{'TOTAL':32s} {total/1e6:10.2f}M {total*4/1024**2:10.1f}")

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu

This is a bit higher than what we expected.

- Actual memory used by the model and additional overhead: 661 − 153 = 508 MiB
- Estimated weight memory: 474.7 MiB
- Difference: 508 − 474.7 = 33.3 MiB

This additional ~33 MiB could be due to model buffers, memory allocator overhead, fragmentation, and additional CUDA library initialization.

For now, I think we can move on rather than worry about these extra 33 MiB.

## Activations & Input Sequences

Going forward, we'll use TransformerLens to inspect the activations inside GPT-2.

Let's first clean up the memory occupied by our Hugging Face model and initialize GPT-2 using TransformerLens.

TransformerLens was originally developed by Neel Nanda for mechanistic

1.   List item
2.   List item

interpretability. It allows us to attach hooks to different parts of the Transformer and capture intermediate activations during the forward pass.

Mechanistic interpretability is a topic for another day. For now, you can treat TransformerLens as a black box. We'll just use its hooks to see how activations are created and how much memory they occupy.

In [ ]:
# lets clean up the memory , Going forward we will use TranfomerLens for hook and see the activations
import gc

del model

gc.collect();
torch.cuda.empty_cache()

In [ ]:
# setup transformer lens
%pip install "transformer-lens<4.0" circuitsvis

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu

Trasformer lens allow caputer all the activation via a cache object , using cache object we can access every weight, activation and gradient that created during the training process.

In [ ]:
import gc, torch
from transformer_lens import HookedTransformer
#imports
import torch
from jaxtyping import Float , Int
from huggingface_hub import hf_hub_download
from transformer_lens import HookedTransformer, HookedTransformerConfig
from transformer_lens.hook_points import HookPoint

# init the gpt2 with hooked transformer
model = HookedTransformer.from_pretrained("gpt2", device="cuda").eval()



In [ ]:
# a helper method to calcuate the parametes and its sizes
from collections import defaultdict
MIB = 1024**2

def tl_grouped(model):
    g = defaultdict(lambda: {"params": 0, "bytes": 0})
    for name, p in model.named_parameters():
        if   name == "embed.W_E":             k = "token embedding (W_E)"
        elif name == "pos_embed.W_pos":       k = "position embedding (W_pos)"
        elif name == "unembed.W_U":           k = "unembedding (W_U)  ← UNTIED EXTRA"
        elif name == "unembed.b_U":           k = "unembedding bias (b_U)"
        elif name.startswith("ln_final"):     k = "LayerNorm (final)"
        elif name.startswith("blocks."):
            rest = ".".join(name.split(".")[2:])
            if   rest.startswith("attn.W"):  k = "attention W (per-layer sum)"
            elif rest.startswith("attn.b"):  k = "attention b (per-layer sum)"
            elif rest.startswith("mlp.W"):   k = "MLP W (per-layer sum)"
            elif rest.startswith("mlp.b"):   k = "MLP b (per-layer sum)"
            elif rest.startswith("ln"):      k = "LayerNorm (per-layer sum)"
            else:                            k = "other: " + rest
        else:                                k = "other: " + name
        g[k]["params"] += p.numel()
        g[k]["bytes"]  += p.numel() * p.element_size()
    return g

groups = tl_grouped(model)
total_p = sum(v["params"] for v in groups.values())
total_b = sum(v["bytes"]  for v in groups.values())

print(f"\n{'component':38s} {'params':>10s} {'FP32 MiB':>10s}")
print("-" * 60)
for k, v in groups.items():
    print(f"{k:38s} {v['params']/1e6:9.2f}M {v['bytes']/MIB:10.1f}")
print("-" * 60)
print(f"{'TOTAL (TL)':38s} {total_p/1e6:9.2f}M {total_b/MIB:10.1f}")


In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu

### Why Is TransformerLens Using More Memory?

Wait a minute. We just loaded the same GPT-2 model, but this time we have 163.05M parameters instead of 124.44M.

Where did these extra parameters come from?

If we look at the parameter breakdown, TransformerLens maintains a separate unembedding matrix (`W_U`), which occupies another 147.2 MiB.

In the Hugging Face implementation, the token embedding and output projection share the same weights. In our TransformerLens configuration, they are stored separately.

This explains almost all of the additional parameter memory.

For the rest of this experiment, we'll use 622 MiB as our calculated model weight memory and 809 MiB as our current device memory reading.

Now, let's move on to activations.

## Let's Run a Forward Pass

We have loaded the model, but we haven't created any input-dependent activations yet.

Let's start with a batch size of 1 and a sequence length of 1024. We'll generate random token IDs since the actual text doesn't matter for this memory experiment.

We'll pass these tokens through GPT-2 using run_with_cache(). This gives us the logits and a cache containing intermediate activations captured during the forward pass.

Using this cache, we can inspect individual tensors, their shapes, data types, and memory consumption.

In [ ]:
batch_size = 1
seq_len = 1024

tokens = torch.randint(
    low=0,
    high=model.cfg.d_vocab,
    size=(batch_size, seq_len),
    device=model.cfg.device
)

# we pass the 1024 length tokens to the modle , this will return the logits and cache object
# which we will have weights , activation , gradients and optimizer state , we can monitor
# mesures.

logits , cache = model.run_with_cache(tokens, remove_batch_dim=True)



In [ ]:
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/balaramaraju/colab/main/cache_mem_profiler.py",
    "cache_mem_profiler.py",
)

In [ ]:
from cache_mem_profiler import CacheMemoryProfiler
print("loaded:", CacheMemoryProfiler)

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu

We haven't calculated a single gradient or initialized any optimizer states yet, but our GPU memory usage has already jumped from 809 MiB to 4,547 MiB.

That's an additional 3,738 MiB just after running the forward pass with activation caching enabled.

What exactly did the model create during this forward pass?

Instead of looking at all 12 Transformer blocks at once, let's start with the first block and inspect its individual tensors.

I've written a small memory profiler to group these tensors and calculate their sizes.

In [ ]:
prof = CacheMemoryProfiler(cache.cache_dict, n_layers=model.cfg.n_layers, model=model)
#prof.print_table()                          # everything
prof.print_table(layer=0)                   # layer 0 only
import pprint
pprint.pprint(prof.summary())  # {'weight':…, 'activation':…, 'total':…}

prof.plot_histograms()    # all: 4-kind + stacked bar + 12 per-layer hists


### Gradients and Backpropagation

So far, we have the model weights and the activations captured during the forward pass.

Now let's calculate the loss and run the backward pass.

As per our theoretical calculation, each parameter's gradient should have the same shape as the parameter itself. Since we are using FP32, we expect the gradients to occupy another ~622 MiB.

Let's run the backward pass and use our memory profiler to verify this.

In [ ]:
# Build a scalar loss: predict next token
loss = model.loss_fn(logits, tokens, per_token=False)   # scalar
print("loss:", loss.item(), "shape:", loss.shape)

loss.backward()

prof = CacheMemoryProfiler(cache.cache_dict, n_layers=model.cfg.n_layers, model=model)
prof.print_table(layer=0)                   # layer 0 only
import pprint
pprint.pprint(prof.summary())
prof.plot_histograms()

As expected, the gradients occupy exactly the same amount of memory as the model weights.

If you look at the individual tensors, you can see that each weight and its corresponding gradient have the same shape, data type, and memory footprint.

| Component | Before backward (MiB) | After backward (MiB) |
|---|---:|---:|
| Weights | 621.98 | 621.98 |
| Activations | 2044.41 | 2044.41 |
| Gradients | 0 | 621.98 |
| **Total** | **2666.40** | **3288.38** |

Our calculation matches what we see in the profiler.

But wait — why are the activations still in memory?

Normally, PyTorch releases the tensors saved for backward as they are no longer needed. In our case, the TransformerLens cache is still holding references to the captured activations.

This is a side effect of our experiment, not a requirement of the training process.

We'll clean up the cache later and see how much memory we get back.

## Optimizer States

We now have the model weights and their gradients. The next step is to update the weights.

We'll use AdamW, a commonly used optimizer for training Transformers.

AdamW maintains two additional values for each trainable parameter:

- First moment — a moving average of the gradients.
- Second moment — a moving average of the squared gradients.

Since we are using FP32, we expect the optimizer states to occupy roughly twice the memory of the model weights.

`622 × 2 = 1244 MiB`

Let's initialize the optimizer, run the optimizer step, and verify this using our memory profiler.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

model.optimizer = optimizer
optimizer.step()
prof = CacheMemoryProfiler(cache.cache_dict, n_layers=model.cfg.n_layers, model=model)
prof.print_table(layer=0)                   # layer 0 only
import pprint
pprint.pprint(prof.summary())
prof.plot_histograms()

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu

In [ ]:
import torch, gc

def print_gpu_mem(label):
    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved  = torch.cuda.memory_reserved() / 1024**2
    print(f"[{label}] Allocated: {allocated:.1f} MB | Reserved: {reserved:.1f} MB")

print_gpu_mem("Before del cache")
prof.raw_cache = None          # <- THE missing step
prof.entries.clear()
del prof

del cache
del logits
del loss
gc.collect()
torch.cuda.empty_cache()

print_gpu_mem("After del cache & empty_cache")

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu

### The breakdown of what is still alive in your GPU right now:

* Weights (model.parameters()): ~622 MB
* Gradients (param.grad): ~622 MB
* Optimizer State (optimizer.state): ~1244 MB
* CUDA Context / overhead: ~100 MB

Total = ~2588 MB

### Why is nvidia-smi 3361 MiB?
It is the sum of three things:

~ 2588 MB: Your live tensors.

630 MB: PyTorch's Caching Allocator (Fragmentation). When you deleted the 2044 MB cache, PyTorch freed those blocks. However, the cache was made of tensors of varying sizes (48 MB, 12 MB, 3 MB). Because they were scattered, PyTorch couldn't return them all to the OS. It kept them in its internal pool ("Reserved") so future tensors can reuse them quickly.

~140 MB: The CUDA Context (cuBLAS handles, kernel binaries, and the PyTorch C++ backend). This is permanent and never leaves until you restart the kernel.



In [ ]:
# final clean up
import gc
import torch

# 1. Drop gradients
model.zero_grad(set_to_none=True)

# 2. Drop optimizer and model
if hasattr(model, "optimizer"):
    model.optimizer = None
del optimizer
del model

# 3. Force PyTorch to release the fragmented pool
gc.collect()
torch.cuda.empty_cache()

In [ ]:
print(f"Allocated: {torch.cuda.memory_allocated()/1024**2:.1f} MB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1024**2:.1f} MB")

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu

### Final Cleanup

We are done with our experiment. Let's clear the gradients, delete the optimizer and model, and release the unused memory from PyTorch's caching allocator.

After cleanup:

Allocated: 16.3 MiB

Reserved: 42 MiB

nvidia-smi: 181 MiB

We started with 153 MiB after initializing CUDA, went all the way up to 5741 MiB during our experiment, and are now back to 181 MiB.

Almost back to our good old state before we initialized the model!